# 📧 Reacher — Check If Email Exists

This notebook installs and runs **[check-if-email-exists](https://github.com/reacherhq/check-if-email-exists)** — a free, open-source tool that verifies whether an email address is real, without sending any email.

---

### ▶️ How to use
Click **Runtime → Run all** from the menu above to install and test automatically.

---

### 📊 What the results mean

| Result | Meaning |
|--------|---------|
| ✅ SAFE | Email is real and accepting messages |
| ⚠️ RISKY | Email exists but may bounce |
| ❌ INVALID | Email does not exist |
| ❓ UNKNOWN | Could not verify (network limitation) |

In [ ]:
# ============================================================
#  STEP 1 — Download & Install the Tool
# ============================================================

import urllib.request, json, subprocess, os

print('=' * 55)
print('  📦 STEP 1 of 2 — Installing the tool...')
print('=' * 55)

api_url = 'https://api.github.com/repos/reacherhq/check-if-email-exists/releases/latest'
with urllib.request.urlopen(api_url) as r:
    data = json.loads(r.read())

version = data['tag_name']
print(f'  ✅ Latest version found: {version}')

download_url = None
filename = None
for asset in data['assets']:
    if 'linux' in asset['name'] and 'musl' in asset['name']:
        download_url = asset['browser_download_url']
        filename = asset['name']
        break

if not download_url:
    print('  ❌ Could not find a Linux binary. Please check the releases page manually.')
    raise SystemExit

print(f'  ✅ Found binary: {filename}')

urllib.request.urlretrieve(download_url, 'ciee.tar.gz')
print('  ✅ Download complete!')

result = subprocess.run(['tar', '-xzf', 'ciee.tar.gz'], capture_output=True, text=True)
if result.returncode == 0:
    print('  ✅ Extraction complete!')
else:
    print('  ❌ Extraction failed:', result.stderr)
    raise SystemExit

os.chmod('check_if_email_exists', 0o755)
print('  ✅ Tool is ready!')
print()

In [ ]:
# ============================================================
#  STEP 2 — Auto-Test with a Sample Email
# ============================================================

import subprocess, json

TEST_EMAIL = 'test@gmail.com'

print('=' * 55)
print('  🧪 STEP 2 of 2 — Running automatic test...')
print('=' * 55)
print(f'  🔍 Testing with sample email: {TEST_EMAIL}')
print()

result = subprocess.run(['./check_if_email_exists', TEST_EMAIL], capture_output=True, text=True)

try:
    out = json.loads(result.stdout)
    reachable = out.get('is_reachable', 'unknown').upper()
    emoji = {'SAFE': '✅', 'RISKY': '⚠️', 'INVALID': '❌', 'UNKNOWN': '❓'}.get(reachable, '❓')
    print(f'  {emoji}  Overall Result : {reachable}')
    print(f'  📝  Valid Format  : {out.get("syntax", {}).get("is_valid_syntax", "N/A")}')
    print(f'  🌐  Domain Exists : {out.get("mx", {}).get("accepts_mail", "N/A")}')
    print(f'  📨  Deliverable   : {out.get("smtp", {}).get("is_deliverable", "N/A")}')
    print(f'  🚫  Disabled Acct : {out.get("smtp", {}).get("is_disabled", "N/A")}')
    print()
    print('=' * 55)
    print('  🎉 All done! Installation verified and working.')
    print()
    print('  💡 To check your own email, add a new cell and run:')
    print()
    print('     import subprocess, json')
    print('     r = subprocess.run(["./check_if_email_exists", "you@example.com"],')
    print('                        capture_output=True, text=True)')
    print('     print(r.stdout)')
    print('=' * 55)
except Exception:
    print('  ⚠️  Could not parse result. Raw output:')
    print(result.stdout or result.stderr)